# DeltaGrad hyperparameter tuning

Pick one benchmark task, run Optuna over the optimizers you care about, and write
the winning kwargs to `best_params/<task>/<optimizer>.pkl` -- exactly where
`python -m experiments.run_task --task <task> --optimizer <optimizer> --use-tuned`
reads them back from. Tuning only: no benchmark figures come out of here, that is
`notebooks/analyze_results.ipynb`'s job.

The machinery lives in `experiments/tune_hyperparams.py` -- the search spaces, the
train/validation split, the pruning callback, the save format. This notebook only
drives it, so changing a search space in the module changes what runs here too.

| Section | What it does |
|---|---|
| 1 Setup | find the repo, import the toolkit |
| 2 Catalogue | what is tunable, and what has already been tuned |
| 3 Choose | the one cell you edit: task, optimizers, budget |
| 4 Data | build the train/validation split once, shared by every trial |
| 5 Budget | time the real machine, estimate the sweep before committing to it |
| 6 Run | the studies -- resumable, interrupt-safe, saved as they finish |
| 7 Saved | what landed on disk, and the command that consumes it |
| 8 Inspect | best trial, optimization history, which knobs actually mattered |
| 9 Coverage | every task/optimizer pair tuned so far |

**Scoring never touches the test set.** `train_val_loaders` carves a validation
split out of the *training* set and drops the task's test loader, so tuned
hyperparameters cannot leak test information into the benchmark that later reports
on them.

## 1. Setup

Finds the repo (locally: walks up from the working directory; on Colab: pulls or
clones it onto local disk, per `notebooks/colab_bootstrap.ipynb` -- git and SQLite
I/O over the Drive FUSE mount are slow enough to look like a hang, so
`optuna_studies/` and `best_params/` must live on `/content`, not Drive). Unlike
the analysis notebook this one **trains models**, so a GPU runtime is worth having
for anything CIFAR-sized.

**Security note:** the GitHub token below is embedded in the git remote URL for
this session only. Never `print()` it or run `!git remote -v` -- either would leak
it in plaintext into a cell output, which persists if you save/share the notebook.

In [ ]:
import os
import subprocess
import sys

# Colab only -- fill in once. Matches notebooks/colab_bootstrap.ipynb so both
# notebooks share one clone/push convention.
GITHUB_REPO = "xandasoneill/deltagrad_optimizer"
GIT_USER_NAME = "xandasoneill"
GIT_USER_EMAIL = "xandas.oneill@gmail.com"


def _find_repo_root(start=None):
    """Nearest ancestor directory containing experiments/configs.py."""
    path = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(path, "experiments", "configs.py")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return None
        path = parent


REPO_ROOT = _find_repo_root()
IN_COLAB = "google.colab" in sys.modules

if REPO_ROOT is None and IN_COLAB:
    from google.colab import userdata

    REPO_ROOT = "/content/deltagrad_optimizer"
    # Passed as an argv list rather than an IPython `!` line so the token never
    # gets echoed into this notebook's saved output.
    remote = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{GITHUB_REPO}.git"
    if os.path.isdir(os.path.join(REPO_ROOT, ".git")):
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "origin", "master"], check=False)
    else:
        subprocess.run(["git", "clone", remote, REPO_ROOT], check=True)
    subprocess.run(["git", "-C", REPO_ROOT, "config", "user.email", GIT_USER_EMAIL], check=True)
    subprocess.run(["git", "-C", REPO_ROOT, "config", "user.name", GIT_USER_NAME], check=True)

if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo -- run this notebook from inside "
                       "the DeltaGrad checkout, or set REPO_ROOT by hand.")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

if IN_COLAB:
    # Not preinstalled on Colab -- this is what "ModuleNotFoundError: optuna"
    # comes from if skipped. joblib/seaborn/scipy are pulled in the same pass.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(REPO_ROOT, "requirements.txt")], check=True)

print("Repo root:", REPO_ROOT)

### 1.1 Toolkit

Imported, not reimplemented: `objective`, `search_space_for` and `train_val_loaders`
are the same functions the CLI runs.

`quiet()` swallows the per-epoch progress `deltagrad/training.py` prints -- 25
trials x 50 epochs is well over a thousand lines of scrollback around the one
number you are watching -- and hands back the real stdout so section 6 can still
print its one line per trial. Set `VERBOSE_TRAINING = True` to keep the raw stream,
which is what you want when a single trial is misbehaving and you need to watch it
epoch by epoch.

In [ ]:
import contextlib
import glob
import io
import time

import joblib
import numpy as np
import optuna
import pandas as pd
import torch

from deltagrad.models import vae_loss
from experiments.configs import TASK_REGISTRY, OPTIMIZER_KEYS
from experiments.final_benchmark import build_optimizer
from experiments.tune_hyperparams import (
    finalize_kwargs,
    save_study,
    search_space_for,
    sqlite_storage,
    train_val_loaders,
    tune_optimizer,
)

VERBOSE_TRAINING = False

# Optuna's INFO logging repeats each trial in a wordier form than section 6's
# callback, so it stays at WARNING -- real problems still surface.
optuna.logging.set_verbosity(optuna.logging.WARNING)


@contextlib.contextmanager
def quiet():
    """Suppresses training's per-epoch prints, yielding the still-live stdout so a
    caller can deliberately print through the gag."""
    if VERBOSE_TRAINING:
        yield sys.stdout
    else:
        console = sys.stdout
        with contextlib.redirect_stdout(io.StringIO()):
            yield console


def tuned_inventory(root="best_params"):
    """One row per best_params/<task>/<optimizer>.pkl already on disk."""
    rows = []
    for path in sorted(glob.glob(os.path.join(root, "*", "*.pkl"))):
        payload = joblib.load(path)
        if not isinstance(payload, dict) or "best_params" not in payload:
            continue  # a hand-written kwargs pkl, or the pre-reorg flat layout
        rows.append({
            "task": payload["task"],
            "optimizer": payload["optimizer"],
            "best_value": payload["best_value"],
            "direction": payload["direction"],
            "trials": payload["n_trials"],
            "epochs": payload["epochs"],
            "batch": payload["batch_size"],
            "tuned_params": payload["tuned_params"],
        })
    return pd.DataFrame(rows)

## 2. What is tunable, and what is already tuned

The first table is every task in `experiments/configs.py::TASK_REGISTRY` at its
full-scale settings -- `epochs x batch` is what one trial costs by default, which is
what section 5 turns into minutes. The second is what has already been tuned;
re-running a pair that appears there overwrites it.

In [ ]:
catalogue = pd.DataFrame([{
    "task": name,
    "type": cfg.task_type,
    "model": cfg.model_cls.__name__,
    "epochs": cfg.epochs,
    "batch": cfg.batch_size,
    "objective": "recon. loss (minimize)" if cfg.task_type == "vae" else "test acc % (maximize)",
    "lr_schedule": "yes" if cfg.lr_scheduler_fn else "-",
} for name, cfg in sorted(TASK_REGISTRY.items())])
display(catalogue)

already = tuned_inventory()
if already.empty:
    print("\nNothing tuned yet -- no best_params/<task>/<optimizer>.pkl on disk.")
else:
    print("\nAlready tuned:")
    display(already.drop(columns=["tuned_params"]))

## 3. Choose the experiment

The only cell you normally edit; everything downstream reads it.

- **`TASK`** -- one key from the catalogue above.
- **`OPTIMIZERS`** -- one study each, run in sequence. `["windowed", "ema", "adam"]`
  is the usual comparison; `OPTIMIZER_KEYS` tunes all seven.
- **`N_TRIALS`** -- total budget per optimizer. 20-30 is a sensible floor for the
  four-parameter spaces; 10 is a sanity check, not a search.
- **`EPOCHS`** -- `None` means the task's own count. This is the main lever on cost,
  with one honest caveat: the objective scores the **final** epoch, so tuning at 15
  epochs finds what wins *at 15 epochs* -- typically a larger learning rate than the
  50-epoch benchmark actually wants.
- **`BATCH_SIZE`** -- `None` means the task's own. Change it only if the benchmark
  will use the changed value too; a learning rate tuned at one batch size does not
  transfer to another.
- **`FIXED`** -- kwargs pinned instead of searched. One flat dict applies to every
  optimizer (`{"weight_decay": 1e-4}`); nest it by optimizer when a pin only makes
  sense for one of them (`{"ema": {"r_transform": "sigmoid"}}` -- `adam` would reject
  `r_transform` outright). For `ema` the transform also decides which shape
  parameters join the space (`gamma` for `exp`, `tau`/`s` for `sigmoid`, ...), since
  each transform ignores the others.
- **`RESUME`** -- keep the studies in `optuna_studies/<task>/tuning.db`. Leave it on:
  a dropped Colab session or a Ctrl-C then costs you the trial in flight instead of
  the whole sweep. Re-running section 6 tops each study up to `N_TRIALS` rather than
  running another `N_TRIALS`.
- **`SEED`** -- seeds model init, the train/val split, batch order *and* the TPE
  sampler, so trials differ by hyperparameters and nothing else.
- **`SMOKE`** -- tiny subset, 1 epoch, 2 trials. Proves the wiring end to end in
  under a minute; the numbers it produces are meaningless as hyperparameters.

In [ ]:
TASK = "mnist_logreg"
# ema is excluded here -- it gets its own dedicated sweep in section 6b below,
# one study per R-transform, since a transform isn't something Optuna searches
# within a single study (each transform reads different shape parameters).
OPTIMIZERS = [key for key in OPTIMIZER_KEYS if key != "ema"]

N_TRIALS = 25
EPOCHS = 45         # None -> the task's own epoch count
BATCH_SIZE = 128      # None -> the task's own batch size

FIXED = {}             # flat, or per-optimizer: {"ema": {"r_transform": "sigmoid"}}
VAL_FRACTION = 0.1
SEED = 42
RESUME = True
DEVICE = None          # None -> cuda when available, else cpu

# Parallel data-loading workers -- overlaps CPU-side batch fetch (PIL decode
# for MNIST/CIFAR) with GPU compute instead of blocking on it. 0 -> the
# loader's own default (synchronous, single-process). Workers respawn every
# epoch (the loader functions don't set persistent_workers), so this trades
# some worker-startup overhead for the fetch/compute overlap -- a net win
# whenever fetch cost isn't trivial, which for MNIST's per-sample PIL decode
# it is not. 2 matches Colab's typical free-tier vCPU count.
NUM_WORKERS = 2

SMOKE = False

### 3.1 Resolve and check

Fails loudly *before* anything trains, on the mistakes that otherwise surface an
hour in: an unknown task or optimizer, a `FIXED` key the optimizer's constructor
would reject, an `r_transform` that does not exist.

Then prints the space each study will search. Worth reading: anything an optimizer
accepts that is not in its space keeps its default and is **not** searched -- pin it
through `FIXED` if you want a different constant.

In [ ]:
if TASK not in TASK_REGISTRY:
    raise KeyError(f"Unknown task {TASK!r}. Choices: {sorted(TASK_REGISTRY)}")
unknown = [key for key in OPTIMIZERS if key not in OPTIMIZER_KEYS]
if unknown:
    raise KeyError(f"Unknown optimizer(s) {unknown}. Choices: {OPTIMIZER_KEYS}")

config = TASK_REGISTRY[TASK]

N_TRIALS_EFFECTIVE = 2 if SMOKE else N_TRIALS
EPOCHS_EFFECTIVE = EPOCHS if EPOCHS is not None else config.effective_epochs(SMOKE)
BATCH_EFFECTIVE = BATCH_SIZE if BATCH_SIZE is not None else config.effective_batch_size(SMOKE)
LOADER_KWARGS = config.effective_loader_kwargs(SMOKE)
LOADER_KWARGS["num_workers"] = NUM_WORKERS
DEVICE_EFFECTIVE = torch.device(DEVICE or ("cuda" if torch.cuda.is_available() else "cpu"))
STORAGE = sqlite_storage(TASK) if RESUME else None

def fixed_for(optimizer_key):
    """FIXED is either one flat dict for every optimizer, or keyed by optimizer when
    a pin is optimizer-specific. Keys that are all optimizer names mean the second
    (no optimizer kwarg is named after an optimizer, so this cannot collide)."""
    if FIXED and set(FIXED) <= set(OPTIMIZER_KEYS):
        return dict(FIXED.get(optimizer_key, {}))
    return dict(FIXED)


FIXED_FOR, SPACES, BASE_KWARGS = {}, {}, {}
for key in OPTIMIZERS:
    FIXED_FOR[key] = fixed_for(key)
    base = config.optimizer_kwargs_for(key)
    base.update(FIXED_FOR[key])
    BASE_KWARGS[key] = base
    SPACES[key] = search_space_for(key, base)     # raises on an unknown r_transform
    # A typo'd or misapplied FIXED key should fail here, not inside trial 0.
    try:
        build_optimizer(key, torch.nn.Linear(1, 1).parameters(), **finalize_kwargs(key, base))
    except TypeError as error:
        raise TypeError(
            f"FIXED {FIXED_FOR[key]} is not valid for optimizer {key!r}: {error}\n"
            f"Pin it for one optimizer only, e.g. FIXED = {{'{key}': {{...}}}}.") from None

print(f"Task       : {TASK}  ({config.model_cls.__name__}, {config.task_type})")
print(f"Objective  : {'minimize reconstruction loss' if config.task_type == 'vae' else 'maximize test accuracy'}"
      f" at epoch {EPOCHS_EFFECTIVE}, on a {VAL_FRACTION:.0%} validation split")
print(f"Budget     : {N_TRIALS_EFFECTIVE} trials x {EPOCHS_EFFECTIVE} epochs x "
      f"{len(OPTIMIZERS)} optimizer(s)")
print(f"Batch size : {BATCH_EFFECTIVE}")
print(f"Device     : {DEVICE_EFFECTIVE}")
print(f"Workers    : {NUM_WORKERS} data-loading process(es)"
      + (" (synchronous fetch -- the loader's own default)" if NUM_WORKERS == 0 else ""))
print(f"Studies    : {STORAGE or 'in memory -- RESUME is off, so an interrupt loses them'}")
for key, pinned in FIXED_FOR.items():
    if pinned:
        print(f"Pinned     : {key} <- {pinned}")
if SMOKE:
    print("SMOKE      : tiny subset / 2 trials -- a wiring check, not hyperparameters")

print()
for key, space in SPACES.items():
    searched = ", ".join(f"{name} in [{spec[1]}, {spec[2]}]"
                         + ("  (log)" if spec[0] == "float" and spec[3] else "")
                         for name, spec in space.items())
    print(f"  {key:<13} searches {len(space)}: {searched}")
    held = {k: v for k, v in BASE_KWARGS[key].items() if k not in space}
    if held:
        print(f"  {'':<13} holds      {held}")

## 4. Data

Built once here and shared by every trial and every optimizer: the split depends on
neither, and rebuilding it per trial is pure overhead (minutes, for IMDB's
re-tokenization).

The shuffle generator comes back with the loaders so each trial can re-seed it and
see an identical batch sequence. Without that, trials would differ by batch order
as well as by hyperparameters, and the sampler would read that noise as signal.

In [ ]:
with quiet():
    LOADERS = train_val_loaders(config, BATCH_EFFECTIVE, LOADER_KWARGS, VAL_FRACTION, SEED)

TRAIN_LOADER, VAL_LOADER, _ = LOADERS
print(f"{len(TRAIN_LOADER.dataset)} train / {len(VAL_LOADER.dataset)} validation examples "
      f"({VAL_FRACTION:.0%} held out of the training set)")
print(f"{len(TRAIN_LOADER)} batches per epoch at batch size {BATCH_EFFECTIVE}")
print("The task's test set is never loaded here, and never scored during tuning.")

## 5. What will this cost?

Optuna offers no estimate, and a sweep is easy to under-budget by an order of
magnitude. This times a handful of real training steps per optimizer, on the real
device with the real data, and extrapolates over batches x epochs x trials. It also
shows DeltaGrad's per-step overhead against the baselines, which is the same
quantity `experiments/ablation_wallclock.py` measures properly.

Read the total as a **ceiling**: the median pruner kills unpromising trials early,
which typically takes 30-50% off. If it is longer than the time you have, cut
`EPOCHS` before `N_TRIALS` -- fewer, longer trials search worse than shorter ones
(with the caveat in section 3).

In [ ]:
CALIBRATE = True
CALIBRATION_BATCHES = 12


def seconds_per_step(optimizer_key, batches=CALIBRATION_BATCHES):
    """Median seconds per training step for this optimizer, measured on the real
    train loader -- fetch and host->device transfer included, not just compute.
    With num_workers=0 (this loader's default) the fetch is not overlapped with
    compute, so for a cheap model it can dwarf the actual training step; timing
    only zero_grad-through-step would silently understate the real cost by an
    order of magnitude. Consuming batches here is harmless: `objective` re-seeds
    the shuffle generator at the start of every trial."""
    model = config.model_cls(**config.model_kwargs).to(DEVICE_EFFECTIVE)
    optimizer = build_optimizer(optimizer_key, model.parameters(),
                                **finalize_kwargs(optimizer_key, BASE_KWARGS[optimizer_key]))
    criterion = torch.nn.CrossEntropyLoss()

    timings = []
    loader_iter = iter(TRAIN_LOADER)
    for _ in range(batches):
        start = time.time()
        inputs, labels = next(loader_iter)
        inputs = inputs.to(DEVICE_EFFECTIVE)
        optimizer.zero_grad()
        outputs = model(inputs)
        if config.task_type == "vae":
            recon, mu, logvar = outputs
            loss = vae_loss(recon, inputs, mu, logvar)
        else:
            loss = criterion(outputs, labels.to(DEVICE_EFFECTIVE))
        loss.backward()
        optimizer.step()
        if DEVICE_EFFECTIVE.type == "cuda":
            torch.cuda.synchronize()
        timings.append(time.time() - start)
    # Median, not mean: the first step or two absorb CUDA context setup and
    # lazily-allocated optimizer state, which no later step pays again.
    return float(np.median(timings))


if CALIBRATE:
    with quiet():
        step_times = {key: seconds_per_step(key) for key in OPTIMIZERS}

    # The validation pass touches VAL_FRACTION of the data with no backward pass;
    # a third of a training step is the usual rule of thumb for that.
    val_overhead = 1 + VAL_FRACTION / 3
    rows = []
    for key, per_step in step_times.items():
        per_epoch = per_step * len(TRAIN_LOADER) * val_overhead
        rows.append({
            "optimizer": key,
            "ms/step": per_step * 1000,
            "vs fastest": per_step / min(step_times.values()),
            "min/trial": per_epoch * EPOCHS_EFFECTIVE / 60,
            "h/study": per_epoch * EPOCHS_EFFECTIVE * N_TRIALS_EFFECTIVE / 3600,
        })
    estimate = pd.DataFrame(rows).round(3)
    display(estimate)

    total_hours = estimate["h/study"].sum()
    print(f"~{total_hours:.1f} h for the whole sweep on {DEVICE_EFFECTIVE}, before pruning "
          f"(~{total_hours * 0.6:.1f} h if pruning cuts the usual 40%).")
    if DEVICE_EFFECTIVE.type == "cpu" and total_hours > 1:
        print("On CPU that is a long sit -- notebooks/colab_bootstrap.ipynb runs the same "
              "code on a Colab GPU.")
else:
    print("Calibration off -- set CALIBRATE = True for a wall-clock estimate.")

## 6. Run the studies

One study per optimizer, in sequence, printing a line per finished trial: its
value, the incumbent best, elapsed time, and the parameters that produced it.

Each optimizer's best params are written **as soon as its own study finishes**, not
at the end of the sweep, so a three-optimizer run interrupted after the second
still leaves two usable files behind.

Interrupting is safe. Ctrl-C (or Kernel > Interrupt) stops the sweep; with
`RESUME = True` the finished trials are already in the database and re-running this
cell continues from there. Re-running after a *complete* sweep does nothing, since
`N_TRIALS` is a total budget rather than an increment -- raise `N_TRIALS` to search
further.

That holds within a session for free. To carry a half-finished study *between*
machines -- including across Colab sessions, which start from a fresh clone -- commit
`optuna_studies/<task>/tuning.db` too, not just the results.

In [ ]:
def progress_callback(optimizer_key, started_at, console):
    """One compact line per finished trial, printed through quiet()'s gag."""
    def on_trial_end(study, trial):
        elapsed = (time.time() - started_at) / 60
        if trial.state.name != "COMPLETE":
            print(f"  [{optimizer_key}] trial {trial.number:>3} {trial.state.name.lower():<8} "
                  f"| {elapsed:5.1f} min", file=console, flush=True)
            return
        params = ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}"
                           for k, v in trial.params.items())
        marker = "  <- best" if trial.number == study.best_trial.number else ""
        print(f"  [{optimizer_key}] trial {trial.number:>3} value {trial.value:8.4f} "
              f"| best {study.best_value:8.4f} | {elapsed:5.1f} min | {params}{marker}",
              file=console, flush=True)
    return on_trial_end


studies, saved_paths = {}, {}

for optimizer_key in OPTIMIZERS:
    print(f"=== {optimizer_key} ===")
    started_at = time.time()
    try:
        with quiet() as console:
            study, base_kwargs, direction = tune_optimizer(
                config, optimizer_key, N_TRIALS_EFFECTIVE, EPOCHS_EFFECTIVE,
                DEVICE_EFFECTIVE, SEED, FIXED_FOR[optimizer_key], LOADERS, storage=STORAGE,
                callbacks=[progress_callback(optimizer_key, started_at, console)])
    except KeyboardInterrupt:
        print(f"  interrupted after {(time.time() - started_at) / 60:.1f} min"
              + (" -- finished trials are in the database; re-run this cell to continue."
                 if STORAGE else " -- RESUME was off, so this study is gone."))
        break

    studies[optimizer_key] = study
    completed = [t for t in study.trials if t.state.name == "COMPLETE"]
    pruned = [t for t in study.trials if t.state.name == "PRUNED"]
    if not completed:
        print(f"  no trial ran to completion ({len(pruned)} pruned) -- nothing saved.\n")
        continue

    saved_paths[optimizer_key] = save_study(study, config, optimizer_key, base_kwargs,
                                            direction, EPOCHS_EFFECTIVE, BATCH_EFFECTIVE)
    print(f"  {len(completed)} completed, {len(pruned)} pruned in "
          f"{(time.time() - started_at) / 60:.1f} min")
    print(f"  best ({direction}) = {study.best_value:.4f}  ->  {saved_paths[optimizer_key]}\n")

if saved_paths:
    print("Wrote: " + ", ".join(saved_paths.values()))

## 6b. EMA R-transform sweep

DeltaGradEMA's `r_transform` picks which of Sec. 3.2's 6 candidate transforms turns
`S_hat` into `R` (`deltagrad/optimizers/ema.py`) -- `linear`, `exp`, `inverse`,
`power`, `sigmoid`, `zscore`. It is not itself a tunable float/int, and each
transform reads its own shape parameters (`gamma`, `power_p`, `tau`/`s`,
`zscore_k`) that the others ignore, so it cannot join a single Optuna study the way
`lr` or `sigma` do.

Instead this runs **one full study per transform** -- `N_TRIALS_EFFECTIVE` trials
each, `6x` the cost of a single optimizer's study in section 6 -- then keeps only
the winner. That's the study saved to `best_params/mnist_logreg/ema.pkl`, in the
same place and shape `run_task.py --use-tuned` already expects, so nothing
downstream needs to know six studies ran to produce it.

Each transform's study is a separate Optuna study (`{task}/ema_{transform}` in the
same `tuning.db`), so this is resumable and interrupt-safe exactly like section 6.

In [ ]:
# deltagrad/optimizers/ema.py's R_TRANSFORMS keys -- Sec. 3.2's 6 candidates.
EMA_TRANSFORMS = ["linear", "exp", "inverse", "power", "sigmoid", "zscore"]

ema_base_fixed = fixed_for("ema")   # honors FIXED = {"ema": {...}} for anything besides r_transform
ema_studies, ema_best_values = {}, {}

for transform in EMA_TRANSFORMS:
    print(f"=== ema ({transform}) ===")
    fixed_kwargs = {**ema_base_fixed, "r_transform": transform}

    # Same pre-flight check as section 3.1, scoped to this transform's kwargs.
    base = config.optimizer_kwargs_for("ema")
    base.update(fixed_kwargs)
    search_space_for("ema", base)  # raises on an unknown r_transform
    build_optimizer("ema", torch.nn.Linear(1, 1).parameters(),
                    **finalize_kwargs("ema", base))

    started_at = time.time()
    try:
        with quiet() as console:
            study, base_kwargs, direction = tune_optimizer(
                config, "ema", N_TRIALS_EFFECTIVE, EPOCHS_EFFECTIVE, DEVICE_EFFECTIVE, SEED,
                fixed_kwargs, LOADERS, storage=STORAGE, study_name=f"{TASK}/ema_{transform}",
                callbacks=[progress_callback(f"ema:{transform}", started_at, console)])
    except KeyboardInterrupt:
        print(f"  interrupted after {(time.time() - started_at) / 60:.1f} min"
              + (" -- finished trials are in the database; re-run this cell to continue."
                 if STORAGE else " -- RESUME was off, so this study is gone."))
        break

    completed = [t for t in study.trials if t.state.name == "COMPLETE"]
    if not completed:
        print(f"  no trial ran to completion -- skipping {transform}\n")
        continue

    ema_studies[transform] = (study, base_kwargs, direction)
    ema_best_values[transform] = study.best_value
    print(f"  best ({direction}) = {study.best_value:.4f} in "
          f"{(time.time() - started_at) / 60:.1f} min\n")

if not ema_best_values:
    print("No EMA transform produced a completed trial -- nothing saved.")
else:
    direction = next(iter(ema_studies.values()))[2]
    ranked = sorted(ema_best_values, key=ema_best_values.get, reverse=(direction == "maximize"))
    best_transform = ranked[0]

    print("EMA transform comparison:")
    for transform in ranked:
        marker = "  <- best" if transform == best_transform else ""
        print(f"  {transform:<8} {ema_best_values[transform]:.4f}{marker}")

    best_study, best_base_kwargs, direction = ema_studies[best_transform]
    path = save_study(best_study, config, "ema", best_base_kwargs, direction,
                      EPOCHS_EFFECTIVE, BATCH_EFFECTIVE)

    # Feeds sections 7-9 the same way a plain optimizer study would.
    studies["ema"] = best_study
    saved_paths["ema"] = path
    print(f"\nWinning transform: {best_transform}  ->  {path}")

## 7. What was saved

`best_params/<task>/<optimizer>.pkl` holds the constructor-ready kwargs (Adam's
`beta1`/`beta2` already recombined into the `betas` tuple torch expects), the
subset Optuna actually searched, and the provenance needed to judge whether the
numbers still apply: epochs, batch size, trial count, best value.

`optuna_studies/<task>/<optimizer>.pkl` holds the whole study object -- every trial,
for section 8 or for later re-analysis -- alongside `tuning.db` if `RESUME` is on.

In [ ]:
for optimizer_key, path in saved_paths.items():
    payload = joblib.load(path)
    print(path)
    print(f"  best ({payload['direction']}) = {payload['best_value']:.4f} over "
          f"{payload['n_trials']} trials at {payload['epochs']} epochs / batch {payload['batch_size']}")
    print(f"  searched  : {payload['tuned_params']}")
    print(f"  passed to the optimizer: {payload['best_params']}\n")

if saved_paths:
    print("Benchmark them (5 seeded runs each, on the real test set):\n")
    for optimizer_key in saved_paths:
        print(f"  python -m experiments.run_task --task {TASK} "
              f"--optimizer {optimizer_key} --use-tuned")
    print("\nThen read the results with notebooks/analyze_results.ipynb.")
else:
    print("Nothing saved yet -- run section 6.")

## 8. Inspect a study

Optional, and to be read with the trial count in mind: 25 trials describe a small
sample, not a landscape.

- **Optimization history** -- every trial's value with the running best. A best that
  flattens out early means the budget was enough; one still climbing at the last
  trial means it was not.
- **Parameter importances** -- fANOVA over the completed trials. Most useful as a
  negative signal: a parameter with near-zero importance can be pinned through
  `FIXED` next time, freeing budget for the ones that move the objective.

In [ ]:
INSPECT = next(iter(studies), None)   # <- set to any key of `studies` by hand

if INSPECT is None:
    print("No study in memory -- run section 6 first.")
else:
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_theme(style="whitegrid", context="notebook")
    plt.rcParams.update({"figure.dpi": 110, "figure.constrained_layout.use": True,
                         "grid.alpha": 0.35, "grid.linestyle": "--"})

    study = studies[INSPECT]
    minimizing = study.direction.name == "MINIMIZE"
    completed = [t for t in study.trials if t.state.name == "COMPLETE"]

    frame = (study.trials_dataframe(attrs=("number", "value", "state", "params", "duration"))
             .sort_values("value", ascending=minimizing))
    print(f"{INSPECT}: {len(completed)} completed of {len(study.trials)} trials -- top 10")
    display(frame.head(10))

    if completed:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        numbers = [t.number for t in completed]
        values = [t.value for t in completed]
        running = (np.minimum if minimizing else np.maximum).accumulate(values)
        pruned = [t.number for t in study.trials if t.state.name == "PRUNED"]
        for number in pruned:
            axes[0].axvline(number, color="0.88", lw=1, zorder=0)
        axes[0].scatter(numbers, values, s=24, alpha=0.75, color="#7b3fa0", label="trial")
        axes[0].plot(numbers, running, color="#008080", lw=2, label="best so far")
        axes[0].set(xlabel="trial", ylabel="validation objective",
                    title=f"{INSPECT} -- optimization history"
                          + (f" ({len(pruned)} pruned, in grey)" if pruned else ""))
        axes[0].legend()

        # fANOVA needs a few completed trials and at least one parameter that varied.
        try:
            importances = optuna.importance.get_param_importances(study)
            names = list(importances)[::-1]
            axes[1].barh(names, [importances[name] for name in names], color="#008080")
            axes[1].set(xlabel="relative importance",
                        title="which knobs moved the objective")
        except Exception as error:
            axes[1].axis("off")
            axes[1].text(0.5, 0.5, f"importances unavailable:\n{error}",
                         ha="center", va="center", wrap=True, transform=axes[1].transAxes)
        plt.show()

## 9. Tuned so far

Every `best_params/<task>/<optimizer>.pkl` in the repo, including runs from other
sessions and machines. A gap here is a `--use-tuned` benchmark you cannot run yet.

In [ ]:
inventory = tuned_inventory()
if inventory.empty:
    print("best_params/ holds nothing in the current schema yet.")
else:
    display(inventory)
    coverage = (inventory.assign(done=1)
                .pivot_table(index="task", columns="optimizer", values="done", fill_value=0)
                .astype(int))
    print("Coverage (1 = tuned):")
    display(coverage)
    for task, group in inventory.groupby("task"):
        gaps = [key for key in OPTIMIZER_KEYS if key not in group["optimizer"].values]
        if gaps:
            print(f"{task}: not yet tuned -> {', '.join(gaps)}")

## 10. Push results to GitHub

Colab only -- and only relevant if section 1 cloned the repo (local runs already
have their working tree; commit and push those the normal way).

Commits `best_params/` (the tuned kwargs `run_task.py --use-tuned` reads) and
`optuna_studies/` (the full studies, `tuning.db` included -- keeping it means a
future Colab session can `RESUME` this exact sweep instead of starting over) and
pushes straight to `origin master`. The `git diff --cached --quiet` guard skips
the commit, without erroring, if a rerun produced nothing new.

In [ ]:
import time

if not IN_COLAB:
    print("Not on Colab -- nothing to push here. Commit best_params/ and "
          "optuna_studies/ yourself the normal way if you want to keep this run.")
else:
    os.chdir(REPO_ROOT)
    subprocess.run(["git", "-C", REPO_ROOT, "add", "best_params/", "optuna_studies/"], check=True)

    staged = subprocess.run(["git", "-C", REPO_ROOT, "diff", "--cached", "--quiet"])
    if staged.returncode == 0:
        print("Nothing new to commit -- best_params/ and optuna_studies/ already match origin.")
    else:
        commit_msg = f"Colab tuning run ({TASK}): {time.strftime('%Y-%m-%d %H:%M:%S')}"
        subprocess.run(["git", "-C", REPO_ROOT, "commit", "-m", commit_msg], check=True)
        subprocess.run(["git", "-C", REPO_ROOT, "push", "origin", "master"], check=True)
        print(f"Pushed: {commit_msg}")